# 05 - Adversarially trained projection encoder (team method, enc_adv3) under the reproduced setup

Port of the team's encoder projection (`infogan_enc.py`, `train_encoder.py`, `attack_eval.py`, `transfer_eval.py`; findings doc "TMD Encoder Projection - Findings") into the setup that reproduces the paper (notebook 00): the authors' fine-tuning recipe, our searched InfoGAN, and the **authors' own TextAttack fork** for every attack. This makes the encoder's numbers directly comparable to Table 1 and to our reproduced TMD.

**Method (same as the team's):** an encoder `E(t, c) -> z` replaces TMD's best-of-30 random latent search. It is trained with G, Q frozen, in rounds:
`enc_den` (reconstruction + latent cycle + typical-set penalty + synonym-swap denoising) -> `enc_adv` (+ real TextFooler adversarial pairs on train vs vanilla and vs enc_den, + keep-label loss) -> `enc_adv2` (+ TextFooler vs enc_adv) -> `enc_adv3` (+ PWWS vs vanilla and vs enc_adv2). Each round starts from the previous encoder. Projection methods: 3 = TMD, 9 = encoder, 10 = encoder + local search, 11 = TMD with one fixed latent bank (control for randomness).

**What is new here:** adversarial pairs are regenerated against *our* model with the fork; evaluation on the same test sample as the reproduction; exact McNemar tests on per-example logs; transfer tests; and a **control**: adversarial training of only the classifier head on the same pairs (no GAN, no encoder), to check that the manifold is what helps.

**Assumptions to confirm with the team:** loss weights and pair counts below are read from their doc and code defaults, not from their exact commands. Change `W_DEN`, `W_ADV` and `ROUNDS` if they differ.

**Cost (L4, estimates):** pair generation about 2 hours in total (resumable, cached), encoder training minutes per round, evaluation at `N_EVAL=200` roughly an hour with BERT-Attack. Run notebook 00 first (it trains the model and GAN this notebook loads).

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
N_EVAL = 200                       # screening size; 1000 for numbers to put next to Table 1
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # must match notebook 00
ATTACKS = ('textfooler', 'pwws', 'bertattack')    # BERT-Attack is held out: never used to make training pairs
ENC_EPOCHS = 30

# ASSUMED from the team's doc/code defaults -- replace with their exact settings if different
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)
ROUNDS = [  # (new encoder, initialised from, new adversarial pairs: (attack, against, n train texts, seed))
    ('enc_adv',  'enc_den',  [('textfooler', 'vanilla', 1000, 1), ('textfooler', 'enc_den', 1000, 2)]),
    ('enc_adv2', 'enc_adv',  [('textfooler', 'enc_adv', 1000, 3)]),
    ('enc_adv3', 'enc_adv2', [('pwws', 'vanilla', 1000, 4), ('pwws', 'enc_adv2', 1500, 5)]),
]

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)
head = lab.model.classifier

In [ ]:
# Synonym-swapped copy of the train set (30% of words, counter-fitted neighbours) for the denoising loss
aug = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=0.3, seed=0)
E_aug, _ = lab.embeddings('train', 'base', augment_fn=aug, name='train_aug')

encs = {}
def get_enc(name, **kw):
    """Train (or load from Drive) one encoder round. Delete its file to retrain; pairs generated against it
    (advpairs_<name>_*) are then stale too and must be deleted."""
    path = lab._path(f'projenc_{name}_L{gan.latent_dim}C{gan.code_dim}.pt')
    enc = T.ProjEncoder(gan.latent_dim, gan.code_dim).to(T.DEV)
    if os.path.exists(path):
        enc.load_state_dict(torch.load(path)); print('loaded', name)
        return enc.eval()
    print('training', name)
    enc = T.train_proj_encoder(gan, E_tr, Y_tr, E_va, Y_va, head=head, epochs=ENC_EPOCHS, **kw)
    torch.save(enc.state_dict(), path)
    return enc

encs['enc_den'] = get_enc('enc_den', E_aug=E_aug, **W_DEN)

In [ ]:
def victim_for(name):
    if name == 'vanilla':
        return T.Victim(lab, mode='none')
    return T.Victim(lab, gan=gan, enc=encs[name], mode='tmd', proj_mode='encoder')   # method 9

all_pairs = []
for name, init, new_pairs in ROUNDS:
    for attack, against, n, seed in new_pairs:          # successful attacks on TRAIN texts only
        all_pairs.append(T.adversarial_pairs(lab, victim_for(against), against, attack, n, seed))
    adv = tuple(torch.cat([p[j] for p in all_pairs]) for j in range(3))
    print(f'{name}: {len(adv[0])} adversarial training pairs')
    encs[name] = get_enc(name, E_aug=E_aug, adv=adv, init=encs[init], **W_ADV)

In [ ]:
# Cheap check on the full test set: reconstruction loss (RL) and clean accuracy after projection
Et = E_te.to(T.DEV)
print(f"{'projection':28s}{'RL':>8s}{'clean acc':>11s}")
print(f"{'raw (no projection)':28s}{'':>8s}{(head(Et).argmax(-1).cpu() == Y_te).float().mean().item() * 100:>11.2f}")
for name, kw in [('TMD m3', dict(mode='sample', k=BASE_K)), ('TMD fixed bank m11', dict(mode='fixed_bank', k=BASE_K)),
                 ('enc_den m9', dict(mode='encoder', enc=encs['enc_den'])),
                 ('enc_adv3 m9', dict(mode='encoder', enc=encs['enc_adv3'])),
                 ('enc_adv3 m10 (sigma 0.5)', dict(mode='enc_local', enc=encs['enc_adv3'], k=BASE_K, sigma_scale=0.5))]:
    with torch.no_grad():
        th, d = T.project(gan, Et, **kw)
        acc = (head(th).argmax(-1).cpu() == Y_te).float().mean().item() * 100
    print(f'{name:28s}{d.mean().item():>8.3f}{acc:>11.2f}')

## Adaptive attacks (the attacker queries the defense)
Same test sample as notebook 00 (the first `N_EVAL` of its 1000). `need_log=True` keeps one line per example so the McNemar and transfer tests below can pair them.

In [ ]:
victims = {
    'vanilla': T.Victim(lab, mode='none'),
    'baseline': T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K),            # TMD (method 3)
    'enc_adv3_m9': T.Victim(lab, gan=gan, enc=encs['enc_adv3'], mode='tmd', proj_mode='encoder'),
}
for tag, v in victims.items():
    T.evaluate(lab, v, tag, E_te, Y_te, attacks=ATTACKS, n=N_EVAL, need_log=True)
display(T.compare(lab, 'enc_adv3_m9', baseline_tag='baseline', n=N_EVAL))

In [ ]:
# Exact McNemar test, TMD vs enc_adv3 on the same sentences ("robust_only_a" = survived only with TMD)
display(pd.DataFrame([T.mcnemar(lab, 'baseline', 'enc_adv3_m9', a, N_EVAL) for a in ATTACKS]))

## Transfer test (the attack never queries the defense)
Texts that fooled vanilla BERT in the adaptive run above, fed once to each defense. Score masking cannot help here. Stochastic defenses are averaged over 5 passes.

In [ ]:
extra = {
    'tmd_fixed_bank_m11': T.Victim(lab, gan=gan, mode='tmd', proj_mode='fixed_bank', k=BASE_K),
    'enc_den_m9': T.Victim(lab, gan=gan, enc=encs['enc_den'], mode='tmd', proj_mode='encoder'),
    'enc_adv3_m10': T.Victim(lab, gan=gan, enc=encs['enc_adv3'], mode='tmd', proj_mode='enc_local', k=BASE_K, sigma_scale=0.5),
}
for attack in ('textfooler', 'pwws'):
    print('transfer from vanilla,', attack)
    display(T.transfer_eval(lab, {**victims, **extra}, 'vanilla', attack, N_EVAL))

## Control: adversarial training without the manifold
The classifier head alone is trained on the clean train embeddings plus exactly the same adversarial pairs enc_adv3 saw. If it matches enc_adv3, the gains come from adversarial training, not from the GAN or the encoder.

In [ ]:
adv_all = tuple(torch.cat([p[j] for p in all_pairs]) for j in range(3))
ctrl_path = lab._path(f'head_adv_control_n{len(adv_all[0])}.pt')
ctrl_head = torch.nn.Linear(768, lab.cfg['n_cls']).to(T.DEV)
if os.path.exists(ctrl_path):
    ctrl_head.load_state_dict(torch.load(ctrl_path)); ctrl_head.eval()
else:
    ctrl_head = T.train_head_adv(lab, E_tr, Y_tr, adv_all)
    torch.save(ctrl_head.state_dict(), ctrl_path)
ctrl = T.Victim(lab, head=ctrl_head, mode='none')
T.evaluate(lab, ctrl, 'head_adv_control', E_te, Y_te, attacks=ATTACKS, n=N_EVAL, need_log=True)
display(T.compare(lab, 'head_adv_control', baseline_tag='baseline', n=N_EVAL))
display(pd.DataFrame([T.mcnemar(lab, 'head_adv_control', 'enc_adv3_m9', a, N_EVAL) for a in ATTACKS]))
for attack in ('textfooler', 'pwws'):
    print('transfer from vanilla,', attack)
    display(T.transfer_eval(lab, {'head_adv_control': ctrl, 'enc_adv3_m9': victims['enc_adv3_m9']}, 'vanilla', attack, N_EVAL))